<!-- nav -->
[← 3 · Which standard?](03_Which_Standard.ipynb) · [Home](../../README.md) · [5 · One framework, three languages →](05_One_Framework_Three_Languages.ipynb)

# 4 · What names refer to

**The problem.** Two functions each have a local `x`. Renaming one of them by finding every `Identifier` spelled `x`
renames the other too, and an attribute `point.x` that has nothing to do with either. A name means something only in
its scope.

`define(module)` reads a module the way Python does: which **entities** it binds, in which **scopes**, and what each
name refers to.

In [1]:
import { walk } from "@mbse/programs/Framework/Syntax";
import { Definitions, Python314, Syntax as P } from "@mbse/programs/Python";

const source = `def area(w, h):
    x = w * h
    return x


def perimeter(w, h):
    x = 2 * (w + h)
    return x


point.x = area(2, 3)
`;

## Step 1: the naive rename

Renaming every `Identifier` spelled `x` changes three things that aren't one:

In [2]:
const naive = Python314.parse(source);
for (const node of walk(naive)) if (node instanceof P.Identifier && node.spelling === "x") node.spelling = "result";
console.log(Python314.print(naive));

def area(w, h):
    result = w * h
    return result


def perimeter(w, h):
    result = 2 * (w + h)
    return result


point.result = area(2, 3)



## Step 2: entities and scopes

`Definitions.define` gives the module's **program**: the entities, each in its scope, with a qualified name:

In [3]:
const module = Python314.parse(source) as P.Module;
const program = Definitions.define(module);
console.log([...program.entities()].map(String));

[
  "<function area>",
  "<parameter area.w>",
  "<parameter area.h>",
  "<function perimeter>",
  "<parameter perimeter.w>",
  "<parameter perimeter.h>",
  "<variable area.x>",
  "<variable perimeter.x>"
]


There are two variables called `x`, `area.x` and `perimeter.x`, and none for `point.x`: an attribute's name depends
on the object, which isn't known until the program runs.

## Step 3: lookup and referents

`program.lookup(name, at)` resolves a name from a place in the tree, as Python would there. `referents` gives what a
`Name` in the tree refers to:

In [4]:
const area = module.body[0] as P.FunctionDef;
const [x] = program.lookup("x", area.body[0]);
console.log(String(x), x.declarations.map((node: any) => Python314.print(node).trim()));
const names = [...walk(module)].filter((node): node is P.Name => node instanceof P.Name && node.id.spelling === "x");
console.log(names.map((name) => Definitions.referents(program, name).map(String)));

<variable area.x> [ "x" ]


[
  [ "<variable area.x>" ],
  [ "<variable area.x>" ],
  [ "<variable perimeter.x>" ],
  [ "<variable perimeter.x>" ]
]


## Step 4: the safe rename

Rename only the names that refer to `area.x`:

In [5]:
for (const name of names) if (Definitions.referents(program, name).includes(x)) name.id.spelling = "result";
console.log(Python314.print(module));

def area(w, h):
    result = w * h
    return result


def perimeter(w, h):
    x = 2 * (w + h)
    return x


point.x = area(2, 3)



## Step 5: Python's own rules

Python's scoping has corners, and `define` follows them: a name assigned in a function is local to it unless `global`
or `nonlocal` says otherwise, and a method doesn't see its class body's names.

In [6]:
const tricky = Python314.parse(`count = 0
LIMIT = 3


class Counter:
    LIMIT = 10

    def bump(self):
        global count
        count += 1
        return count < LIMIT
`) as P.Module;
const trickyProgram = Definitions.define(tricky);
const bump = (tricky.body[2] as P.ClassDef).body[1];
for (const node of walk(bump)) {
  if (node instanceof P.Name && ["count", "LIMIT"].includes(node.id.spelling)) {
    console.log(node.id.spelling, Definitions.referents(trickyProgram, node).map(String));
  }
}

count [ "<variable count>" ]


count [ "<variable count>" ]


LIMIT [ "<variable LIMIT>" ]


`count` is the module's, because of `global`; `LIMIT` is the module's too, not `Counter.LIMIT`, because lookup from a
method skips the class body.

## Why it works this way

- **Names are resolved, not matched.** A rename, an inline, a dead-code check all need to know what a name refers to.
  Spelling is only the start.
- **Each language's own rules.** Python's scopes are Python's; C++'s namespaces and TypeScript's values and types are
  theirs (case study 5). Each language's `Definitions` follows its standard.
- **What can't be known statically isn't guessed.** Attributes, keyword arguments and imported modules' members depend
  on values, so they find nothing rather than something wrong.

<!-- nav -->
[← 3 · Which standard?](03_Which_Standard.ipynb) · [Home](../../README.md) · [5 · One framework, three languages →](05_One_Framework_Three_Languages.ipynb)